# ESIOS Pull Diagnostics
### One section per indicator — run after each fetch script to verify what was pulled

**Purpose:** Understand exactly what each indicator returns (geo_ids, units, format, range).  
**How to use:** Run the relevant fetch script first, then execute the corresponding section here.

```bash
python src/fetch_prices.py    # then run Section 1
python src/fetch_demand.py    # then run Section 2
python src/fetch_wind.py      # then run Section 3
python src/fetch_solar.py     # then run Section 4
```


## 0 · Setup

In [10]:
import pandas as pd, numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt, warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False,
    "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25,
    "figure.facecolor": "white", "axes.facecolor": "white",
})

import os
DATA = "../../data/raw"
print("Data files available:")
for f in sorted(os.listdir(DATA)):
    path = os.path.join(DATA, f)
    size = os.path.getsize(path) // 1024
    print(f"  {f:<45} {size:>6} KB")


Data files available:
  aemet_barcelona_2015_2026.csv                    222 KB
  electric_consumption_barcelona_2019_2026.csv       1 KB
  electric_consumption_barcelona_economic_sector_2019_2026.csv      5 KB
  electric_consumption_barcelona_electric_price_2015_2026.csv     73 KB
  electric_consumption_barcelona_postal_code_2019_2026.csv      4 KB
  electric_consumption_barcelona_time_interval_2019_2026.csv      8 KB
  esios_demand.csv                                 185 KB
  esios_generation.csv                             732 KB
  esios_prices.csv                                 349 KB
  esios_selfconsumption.csv                       1991 KB


---
## 1 · Prices — indicator 600

**Expected:** 4,138 daily rows, 2015-01-01 → 2026-04-30, Spain mean ~78 €/MWh  
**Columns:** date, price_spain, price_france, price_portugal, spread_fr_es, spread_pt_es


In [11]:
df_p = pd.read_csv(f"{DATA}/esios_prices.csv")
df_p["date"] = pd.to_datetime(df_p["date"])

print("Shape:", df_p.shape)
print("Dates:", df_p.date.min().date(), "→", df_p.date.max().date())
print()
print("Price statistics:")
print(df_p[["price_spain","price_france","price_portugal"]].describe().round(1).to_string())
print()
print("Missing values:", df_p.isna().sum().to_dict())

checks = {
    "rows == 4138":           len(df_p) == 4138,
    "no missing prices":      df_p["price_spain"].isna().sum() == 0,
    "spain mean 30–150":      30 < df_p["price_spain"].mean() < 150,
    "spread_fr_es non-zero":  df_p["spread_fr_es"].std() > 1,
}
print()
for k, v in checks.items():
    print(f"  {'✅' if v else '❌'} {k}")


Shape: (4138, 6)
Dates: 2015-01-01 → 2026-04-30

Price statistics:
       price_spain  price_france  price_portugal
count       4138.0        4138.0          4138.0
mean          78.3          85.8            78.4
std           67.4          96.3            67.4
min            0.4        -163.3             0.1
25%           42.3          34.3            42.4
50%           55.6          49.3            55.6
75%           91.4          89.4            91.7
max          547.4         743.8           542.8

Missing values: {'date': 0, 'price_spain': 0, 'price_france': 0, 'price_portugal': 0, 'spread_fr_es': 0, 'spread_pt_es': 0}

  ✅ rows == 4138
  ✅ no missing prices
  ✅ spain mean 30–150
  ✅ spread_fr_es non-zero


In [12]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle("Prices — indicator 600", fontweight="bold")

axes[0].plot(df_p["date"], df_p["price_spain"], lw=0.6, color="#2C3E6B", label="Spain")
axes[0].plot(df_p["date"], df_p["price_france"], lw=0.6, color="#E67E22", alpha=0.7, label="France")
axes[0].set_title("Daily mean price (€/MWh)")
axes[0].legend(fontsize=9, frameon=False)
axes[0].set_ylabel("€/MWh")

axes[1].hist(df_p["price_spain"], bins=60, color="#2C3E6B", alpha=0.8, edgecolor="white")
axes[1].set_title("Spain price distribution")
axes[1].set_xlabel("€/MWh")
axes[1].set_ylabel("Days")

plt.tight_layout()
plt.savefig("../../data/processed/figures/diag_prices.png", bbox_inches="tight")
plt.show()
print("✓ Prices look clean")


✓ Prices look clean


---
## 2 · Demand — indicator 460

**Expected:** 4,139 daily rows, all years 18,000–35,000 MW mean  
**Known issue:** Format switched from instantaneous MW → cumulative MWh on 2022-05-24  
**Fix applied in fetch_demand.py:** hourly diff() on post-2022-05-24 data


In [19]:
# study day 2022-05-24 in detail by hour
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "src", "esios"))
from fetch_shared import fetch_month, clean_hourly
import pandas as pd

print("Fetching 3-day window around the 2022-05-24 boundary...")
raw = fetch_month(460, "2022-05-22T00:00:00", "2022-05-26T23:59:59")
df  = clean_hourly(raw, "demand")
print(f"  {len(df)} rows returned")

for day_str in ["2022-05-22", "2022-05-23", "2022-05-24", "2022-05-25", "2022-05-26"]:
    day = df[
        (df.date >= pd.Timestamp(day_str, tz="Europe/Madrid")) &
        (df.date <  pd.Timestamp(day_str, tz="Europe/Madrid") + pd.Timedelta(days=1))
    ].sort_values("date").reset_index(drop=True)
    
    flag = "✅" if 18000 < day.demand.mean() < 35000 else "❌"
    print(f"=== {day_str}  ({len(day)} rows)  mean={day.demand.mean():,.0f} MW  {flag}")
    print(f"    min={day.demand.min():,.0f}  max={day.demand.max():,.0f}")
    
    # Show all rows with hour label
    day["hour"] = day["date"].dt.hour
    day["diff_from_prev"] = day["demand"].diff().round(0)
    print(day[["hour", "demand", "diff_from_prev"]].to_string(index=False))
    print()

Fetching 3-day window around the 2022-05-24 boundary...
  120 rows returned
=== 2022-05-22  (24 rows)  mean=23,059 MW  ✅
    min=19,646  max=26,319
 hour  demand  diff_from_prev
    0 23235.0             NaN
    1 22070.0         -1165.0
    2 20976.0         -1094.0
    3 20344.0          -632.0
    4 19793.0          -551.0
    5 19769.0           -24.0
    6 19820.0            51.0
    7 19646.0          -174.0
    8 20460.0           814.0
    9 22137.0          1677.0
   10 23659.0          1522.0
   11 24580.0           921.0
   12 24893.0           313.0
   13 25133.0           240.0
   14 25176.0            43.0
   15 24391.0          -785.0
   16 23831.0          -560.0
   17 23551.0          -280.0
   18 23607.0            56.0
   19 24357.0           750.0
   20 25292.0           935.0
   21 26319.0          1027.0
   22 26160.0          -159.0
   23 24208.0         -1952.0

=== 2022-05-23  (24 rows)  mean=26,913 MW  ✅
    min=20,598  max=30,676
 hour  demand  diff_from_prev

In [20]:
df_d = pd.read_csv(f"{DATA}/esios_demand.csv")
df_d["date"] = pd.to_datetime(df_d["date"])

print("Shape:", df_d.shape)
print("Dates:", df_d.date.min().date(), "→", df_d.date.max().date())
print()
print("By year (mean MW — all should be 18,000–35,000):")
by_yr = df_d.groupby(df_d.date.dt.year)["demand_mean_mw"].mean().round(0)
for yr, v in by_yr.items():
    flag = "✅" if 18000 < v < 35000 else "❌"
    print(f"  {yr}: {v:>9,.0f} MW  {flag}")

print()
n_unique = df_d["demand_mwh_day"].nunique()
mean_mw  = df_d["demand_mean_mw"].mean()
checks = {
    "rows >= 4000":           len(df_d) >= 4000,
    f"unique values > 100 ({n_unique})": n_unique > 100,
    f"mean 22k-32k ({mean_mw:,.0f})":   22000 < mean_mw < 32000,
    "no NaN demand":          df_d["demand_mwh_day"].isna().sum() == 0,
}
for k, v in checks.items():
    print(f"  {'✅' if v else '❌'} {k}")


Shape: (4139, 5)
Dates: 2014-12-31 → 2026-04-30

By year (mean MW — all should be 18,000–35,000):
  2014:    25,438 MW  ✅
  2015:    28,383 MW  ✅
  2016:    28,507 MW  ✅
  2017:    28,861 MW  ✅
  2018:    29,064 MW  ✅
  2019:    28,540 MW  ✅
  2020:    27,062 MW  ✅
  2021:    27,850 MW  ✅
  2022:    26,950 MW  ✅
  2023:    26,174 MW  ✅
  2024:    26,441 MW  ✅
  2025:    27,231 MW  ✅
  2026:    27,538 MW  ✅

  ✅ rows >= 4000
  ✅ unique values > 100 (4119)
  ✅ mean 22k-32k (27,727)
  ✅ no NaN demand


In [21]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle("Demand — indicator 460 (with cumulative fix)", fontweight="bold")
BOUNDARY = pd.Timestamp("2022-05-24")

# 1. Full time series
ax = axes[0, 0]
pre  = df_d[df_d.date < BOUNDARY]
post = df_d[df_d.date >= BOUNDARY]
ax.plot(pre.date,  pre["demand_mean_mw"],  lw=0.7, color="#2C3E6B", label="Pre-2022-05-24 (raw MW)")
ax.plot(post.date, post["demand_mean_mw"], lw=0.7, color="#E67E22", label="Post-2022-05-24 (diff-corrected)")
ax.axvline(BOUNDARY, color="red", lw=1.5, linestyle="--", label="Format boundary")
ax.set_title("Demand mean MW — full series")
ax.legend(fontsize=8, frameon=False)
ax.set_ylabel("MW")

# 2. Zoom on boundary
ax = axes[0, 1]
mask = (df_d.date >= "2022-04-01") & (df_d.date <= "2022-07-01")
zoom = df_d[mask]
ax.plot(zoom.date, zoom["demand_mean_mw"], lw=1.2, color="#2C3E6B", marker="o", markersize=2)
ax.axvline(BOUNDARY, color="red", lw=2, linestyle="--", label="Boundary 2022-05-24")
ax.set_title("Zoom: format transition window")
ax.legend(fontsize=9, frameon=False)
ax.set_ylabel("MW")

# 3. Annual means
ax = axes[1, 0]
annual = df_d.groupby(df_d.date.dt.year)["demand_mean_mw"].mean()
colors = ["#27AE60" if 18000<v<35000 else "#E74C3C" for v in annual.values]
ax.bar(annual.index, annual.values, color=colors, alpha=0.85)
ax.axhline(25000, color="grey", lw=1, linestyle="--", label="Expected ~25k MW")
ax.set_title("Annual mean demand (MW)")
ax.set_ylabel("MW")
ax.legend(fontsize=9, frameon=False)
for x, v in zip(annual.index, annual.values):
    ax.text(x, v+300, f"{v/1000:.0f}k", ha="center", fontsize=8)

# 4. Day-of-week pattern (sanity: weekends should be lower)
ax = axes[1, 1]
df_d["dow"] = df_d.date.dt.dayofweek
dow_mean = df_d.groupby("dow")["demand_mean_mw"].mean()
ax.bar(range(7), dow_mean.values, color="#2C3E6B", alpha=0.8)
ax.set_xticks(range(7))
ax.set_xticklabels(["Mon","Tue","Wed","Thu","Fri","Sat","Sun"])
ax.set_title("Demand by day of week (weekends should be lower)")
ax.set_ylabel("MW")

plt.tight_layout()
plt.savefig("../../data/processed/figures/diag_demand.png", bbox_inches="tight")
plt.show()


---
## 3 · Wind generation — indicator 10288

**Expected:** 4,139 daily rows, mean ~3,300 MW, winter peak, summer trough  
**geo_id=8741 (Península) only** — Canarias (8742) excluded


In [23]:
df_w = pd.read_csv(f"{DATA}/esios_wind.csv")
df_w["date"] = pd.to_datetime(df_w["date"])

print("Shape:", df_w.shape)
print("Dates:", df_w.date.min().date(), "→", df_w.date.max().date())
print()
mean_mw = df_w["wind_mean_mw"].mean()
print(f"Overall mean: {mean_mw:,.0f} MW  (expected 2,000–5,000)")
print(f"Max day:      {df_w['wind_peak_mw'].max():,.0f} MW")
print(f"Missing:      {df_w['wind_mwh_day'].isna().sum()}")

checks = {
    "rows >= 4000":         len(df_w) >= 4000,
    "mean 2k–5k MW":        2000 < mean_mw < 5000,
    "no NaN":               df_w["wind_mwh_day"].isna().sum() == 0,
    "peak hour ≠ noon":     df_w["wind_peak_hour"].mode()[0] not in [12,13],
}
print()
for k, v in checks.items():
    print(f"  {'✅' if v else '❌'} {k}")


Shape: (4139, 5)
Dates: 2014-12-31 → 2026-04-30

Overall mean: 3,310 MW  (expected 2,000–5,000)
Max day:      10,644 MW
Missing:      0

  ✅ rows >= 4000
  ✅ mean 2k–5k MW
  ✅ no NaN
  ✅ peak hour ≠ noon


In [26]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle("Wind — indicator 10288", fontweight="bold")

monthly = df_w.set_index("date").resample("ME")["wind_mean_mw"].mean()
axes[0].fill_between(monthly.index, monthly.values, color="#1A6FA3", alpha=0.7)
axes[0].set_title("Monthly average wind generation (MW)")
axes[0].set_ylabel("MW")

by_month = df_w.groupby(df_w.date.dt.month)["wind_mean_mw"].mean()
axes[1].bar(range(1,13), by_month.values, color="#1A6FA3", alpha=0.85)
axes[1].set_xticks(range(1,13))
axes[1].set_xticklabels(["J","F","M","A","M","J","J","A","S","O","N","D"])
axes[1].set_title("Seasonal wind profile (winter peak expected)")
axes[1].set_ylabel("MW")

plt.tight_layout()
plt.savefig("../../data/processed/figures/diag_wind.png", bbox_inches="tight")
plt.show()


---
## 4 · Utility Solar PV — indicator 10358

**Expected:** Data from Jan 2019 onward; growing trend 2019→2026; peak hour 12–14h  
**Pre-2019:** NaN — indicator not available before 2019. Expected, not a bug.  
**geo_id=8741 (Península) only**

Indicators NOT to use:
- `10289` — broken: decreasing trend from 2023, low values (~2,100 MW mean)
- `1159`  — 403 Forbidden with standard ESIOS token


In [27]:
df_s = pd.read_csv(f"{DATA}/esios_solar.csv")
df_s["date"] = pd.to_datetime(df_s["date"])

n_data = df_s["solar_mwh_day"].notna().sum()
n_nan  = df_s["solar_mwh_day"].isna().sum()
print("Shape:", df_s.shape)
print(f"Days with data: {n_data}  |  NaN (pre-2019): {n_nan}")
print()

solar_post = df_s[df_s["solar_mean_mw"].notna()]
print("By year (mean MW — must be GROWING over time):")
by_yr = solar_post.groupby(solar_post.date.dt.year)["solar_mean_mw"].mean().round(0)
for yr, v in by_yr.items():
    bar = "█" * int(v / 500)
    print(f"  {yr}: {v:>7,.0f} MW  {bar}")

solar_2022 = by_yr.get(2022, 0)
solar_2024 = by_yr.get(2024, 0)
growing = solar_2024 > solar_2022 > 0
print()
print(f"  {'✅ Growing trend confirmed' if growing else '❌ NOT growing — wrong indicator!'}")
print(f"  Peak hour mode: {df_s['solar_peak_hour'].mode()[0]} h (expected 12–14)")


Shape: (4138, 5)
Days with data: 2678  |  NaN (pre-2019): 1460

By year (mean MW — must be GROWING over time):
  2018:   3,214 MW  ██████
  2019:   7,056 MW  ██████████████
  2020:   7,790 MW  ███████████████
  2021:   9,057 MW  ██████████████████
  2022:   9,792 MW  ███████████████████
  2023:  11,131 MW  ██████████████████████
  2024:  11,786 MW  ███████████████████████
  2025:  12,134 MW  ████████████████████████
  2026:  15,480 MW  ██████████████████████████████

  ✅ Growing trend confirmed
  Peak hour mode: 12.0 h (expected 12–14)


In [30]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle("Utility Solar PV — indicator 10358", fontweight="bold")

df_splot = df_s.dropna(subset=["solar_mean_mw"])
monthly = df_splot.set_index("date").resample("ME")["solar_mean_mw"].mean()
axes[0].fill_between(monthly.index, monthly.values, color="#E67E22", alpha=0.7)
axes[0].set_title("Monthly average solar PV (MW) — must grow left→right")
axes[0].set_ylabel("MW")

by_month = df_splot.groupby(df_splot.date.dt.month)["solar_mean_mw"].mean()
axes[1].bar(range(1,13), by_month.reindex(range(1,13), fill_value=0).values,
            color="#E67E22", alpha=0.85)
axes[1].set_xticks(range(1,13))
axes[1].set_xticklabels(["J","F","M","A","M","J","J","A","S","O","N","D"])
axes[1].set_title("Seasonal solar profile (summer peak expected)")
axes[1].set_ylabel("MW")

plt.tight_layout()
plt.savefig("../../data/processed/figures/diag_solar.png", bbox_inches="tight")
plt.show()


---
## 5 · Cross-indicator sanity check

Combines all four outputs to verify:
- Renewables penetration in plausible range (8–50% mean)
- Apr 28, 2025 blackout window shows collapse conditions
- Demand varies seasonally and weekly


In [ ]:
# Load all outputs
df_p = pd.read_csv(f"{DATA}/esios_prices.csv",  parse_dates=["date"])
df_d = pd.read_csv(f"{DATA}/esios_demand.csv",  parse_dates=["date"])
df_w = pd.read_csv(f"{DATA}/esios_wind.csv",    parse_dates=["date"])
df_s = pd.read_csv(f"{DATA}/esios_solar.csv",   parse_dates=["date"])

merged = (df_d
    .merge(df_w.rename(columns={"wind_mwh_day":  "wind_mwh",
                                 "wind_mean_mw":  "wind_mw"})[["date","wind_mwh","wind_mw"]],
           on="date", how="left")
    .merge(df_s.rename(columns={"solar_mwh_day": "solar_mwh",
                                 "solar_mean_mw": "solar_mw"})[["date","solar_mwh","solar_mw"]],
           on="date", how="left")
    .merge(df_p[["date","price_spain"]], on="date", how="left")
)
merged["ren_pen"] = (
    (merged["wind_mwh"].fillna(0) + merged["solar_mwh"].fillna(0))
    / merged["demand_mwh_day"]
)

print("=== CROSS-INDICATOR CHECKS ===")
checks = {
    "demand mean 22k–32k MW":    22000 < merged["demand_mean_mw"].mean() < 32000,
    "wind mean 2k–8k MW":        2000  < merged["wind_mw"].mean()        < 8000,
    "solar (post-2019) mean>1k": merged.loc[merged.date.dt.year>=2019,"solar_mw"].mean() > 1000,
    "renewables pen 8–50%":      8 < merged["ren_pen"].mean()*100 < 50,
    "prices non-negative mean":  merged["price_spain"].mean() > 0,
}
for k, v in checks.items():
    print(f"  {'✅' if v else '❌'} {k}")

print()
print("Blackout window Apr 25 – May 1, 2025:")
bk = merged[(merged.date >= "2025-04-25") & (merged.date <= "2025-05-01")]
print(bk[["date","demand_mean_mw","wind_mw","solar_mw","ren_pen","price_spain"]].to_string(index=False))
